## Name: Abbas Nadir
## Roll No.: A3 34

## **ML Lab 5**

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from sklearn.model_selection import (train_test_split, GridSearchCV, cross_val_score, StratifiedKFold)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report,
                             mean_absolute_error, mean_squared_error, r2_score)

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")

In [ ]:
# Load Dataset
df = pd.read_csv("diabetes.csv")

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
# Check zero values in columns where zero is treated as an invalid measurement
zero_columns = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
zero_counts = (df[zero_columns] == 0).sum()

print("Zero values:")
print(zero_counts)

In [ ]:
df_clean = df.copy()

df_clean[zero_columns] = df_clean[zero_columns].replace(0, np.nan)

for col in zero_columns:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print("Missing values after median imputation:")
print(df_clean.isnull().sum())

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_clean)
plt.title("Box Plot of Diabetes Dataset")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
df_clean.hist(figsize=(12, 10), bins=20)
plt.suptitle("Histogram of Diabetes Dataset")
plt.tight_layout()
plt.show()

In [ ]:
outcome_counts = df_clean["Outcome"].value_counts().sort_index()

plt.figure(figsize=(6, 5))
plt.bar(["No Diabetes", "Diabetes"], outcome_counts.values)
plt.title("Diabetes Outcome Distribution")
plt.xlabel("Outcome")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df_clean, x="Glucose", y="BMI", hue="Outcome")
plt.title("Glucose vs BMI")
plt.xlabel("Glucose")
plt.ylabel("BMI")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
correlation = df_clean.corr()
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
features = ["Glucose", "BloodPressure", "BMI", "Age", "Outcome"]
sns.pairplot(df_clean[features], hue="Outcome")
plt.show()

## KNN Classification

In [ ]:
X = df_clean.drop("Outcome", axis=1)
y = df_clean["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=24, stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

In [ ]:
print("Training distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting distribution:")
print(y_test.value_counts(normalize=True))

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
comparison = pd.DataFrame({
    "Original Mean": X_train.mean(),
    "Original Std": X_train.std(),
    "Scaled Mean": X_train_scaled.mean(axis=0),
    "Scaled Std": X_train_scaled.std(axis=0)
})

display(comparison)

In [ ]:
K = 7

knn = KNeighborsClassifier(n_neighbors=K)
knn.fit(X_train_scaled, y_train)

In [ ]:
y_pred = knn.predict(X_test_scaled)

print("Predictions:")
print(y_pred[:20])

In [ ]:
y_prob = knn.predict_proba(X_test_scaled)[:, 1]
print("Predicted probability for class 1:")
print(y_prob[:20])

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

In [ ]:
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(
    confusion_matrix(y_test, y_pred),
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No Diabetes", "Diabetes"],
    yticklabels=["No Diabetes", "Diabetes"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("KNN Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
k_values = range(1, 26)
train_accuracies = []
test_accuracies = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)

    train_accuracies.append(model.score(X_train_scaled, y_train))
    test_accuracies.append(model.score(X_test_scaled, y_test))

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(k_values, train_accuracies, marker="o", label="Training Accuracy")
plt.plot(k_values, test_accuracies, marker="s", label="Testing Accuracy")
plt.xlabel("Number of Neighbors (K)")
plt.ylabel("Accuracy")
plt.title("KNN Accuracy vs. Number of Neighbors")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
best_k = list(k_values)[int(np.argmax(test_accuracies))]
best_accuracy = max(test_accuracies)

print(f"Best K value: {best_k}")
print(f"Best test accuracy: {best_accuracy:.4f}")

In [ ]:
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=24)
cv_scores = []

# Scaling is placed inside the pipeline so each fold learns its own scaler.
for k in k_values:
    cv_model = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k))
    ])
    scores = cross_val_score(cv_model, X_train, y_train, cv=cv, scoring="accuracy")
    cv_scores.append(scores.mean())

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(list(k_values), cv_scores, marker="o")
plt.xlabel("Number of Neighbors (K)")
plt.ylabel("Cross-Validation Accuracy")
plt.title("KNN Cross-Validation Accuracy vs. K")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
best_cv_k = list(k_values)[int(np.argmax(cv_scores))]
print(f"Best K value from cross-validation: {best_cv_k}")
print(f"Best mean CV accuracy: {max(cv_scores):.4f}")

In [ ]:
param_grid = {
    "knn__n_neighbors": list(range(3, 18, 2)),
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["minkowski"],
    "knn__p": [1, 2]
}

knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])

grid_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
print("Grid search completed.")

In [ ]:
print("Best Parameters:", grid_search.best_params_)
print(f"Best Cross-Validation Accuracy: {grid_search.best_score_:.4f}")

In [ ]:
best_knn = grid_search.best_estimator_
best_y_pred = best_knn.predict(X_test)

print(f"Test Accuracy: {accuracy_score(y_test, best_y_pred):.4f}")
print(classification_report(y_test, best_y_pred, zero_division=0))

## KNN Regression

In [ ]:
# Predict BMI as a continuous value using the other available features.
X_reg = df_clean.drop(columns=["BMI", "Outcome"])
y_reg = df_clean["BMI"]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=24
)

In [ ]:
knn_reg = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor(n_neighbors=7, weights="distance"))
])

knn_reg.fit(X_train_reg, y_train_reg)
y_pred_reg = knn_reg.predict(X_test_reg)

In [ ]:
mae = mean_absolute_error(y_test_reg, y_pred_reg)
mse = mean_squared_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_reg, y_pred_reg)

print(f"Mean Absolute Error: {mae:.4f}")
print(f"Mean Squared Error: {mse:.4f}")
print(f"Root Mean Squared Error: {rmse:.4f}")
print(f"R2 Score: {r2:.4f}")

In [ ]:
regression_comparison = pd.DataFrame({
    "Actual BMI": y_test_reg.iloc[:10].to_numpy(),
    "Predicted BMI": y_pred_reg[:10]
})

display(regression_comparison)

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test_reg, y_pred_reg, alpha=0.7)

line_min = min(y_test_reg.min(), y_pred_reg.min())
line_max = max(y_test_reg.max(), y_pred_reg.max())
plt.plot([line_min, line_max], [line_min, line_max], linestyle="--")

plt.xlabel("Actual BMI")
plt.ylabel("Predicted BMI")
plt.title("KNN Regression: Actual vs Predicted BMI")
plt.tight_layout()
plt.show()